# VoiceGuard — train the AASIST detector (Colab)

Full pipeline: HF data → content-matched MMS-TTS fakes → manifests → (RawBoost) →
fine-tune wav2vec2 + AASIST → evaluate. Output: `aasist_indicw2v.pt` to download into
`backend/models/` locally.

**Runtime → Change runtime type → GPU.** Kaggle (`train_kaggle.ipynb`) is more reliable for
the long run; use this for shorter experiments. Mount Drive if you want the checkpoint to
survive a disconnect.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime → GPU'
print(torch.cuda.get_device_name(0))
!pip -q install -U 'transformers>=4.44' 'datasets>=2.20' huggingface_hub soundfile librosa pyyaml scipy

In [ ]:
REPO_URL = ''   # <-- your GitHub clone URL
HF_TOKEN = ''   # <-- optional: for ai4bharat/indicwav2vec-hindi (accept its licence first)
import os
assert REPO_URL, 'Set REPO_URL (or upload a repo zip and unzip it)'
!git clone --depth 1 {REPO_URL} voiceguard
%cd voiceguard
if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN
!ls

In [ ]:
FRONTEND = 'facebook/wav2vec2-xls-r-300m'   # or ai4bharat/indicwav2vec-hindi (needs HF_TOKEN)
EPOCHS   = 20
UNFREEZE = 6          # frontend frozen for N epochs then fine-tuned
LOSS     = 'oc_softmax'
FAKE_N   = 1200
LIMIT    = None       # e.g. 400 for a smoke run

import yaml, pathlib
p = pathlib.Path('training/config_train.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['device'] = 'cuda'
cfg['frontend'].update(model_id=FRONTEND, layer=-1, stage2_unfreeze_epoch=UNFREEZE)
cfg['loss']['name'] = LOSS
cfg['epochs'] = EPOCHS
cfg['num_workers'] = 2
p.write_text(yaml.safe_dump(cfg, sort_keys=False)); print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
args = f'--config training/config_train.yaml --fake-n {FAKE_N} --epochs {EPOCHS}'
if LIMIT: args += f' --limit {LIMIT}'
!python -m training.pipeline_run {args}

In [ ]:
from google.colab import files
files.download('backend/models/aasist_indicw2v.pt')